# GNARL на TSP: обучение на одном размере графа, инференс на больших

Этот ноутбук — отдельный, самодостаточный эксперимент поверх реализации из
[`gnarl/`](https://github.com/artlvruran/reasoning/tree/gnarl) (статья
*"Tackling GNARLy Problems: Graph Neural Algorithmic Reasoning Reimagined
through Reinforcement Learning"*, Schutz et al.). В отличие от
`gnarl_experiments.ipynb`, где TSP обучался на смеси размеров `{6,8,10}`,
здесь модель обучается **только на графах одного размера** (`|V| = 10`), а
инференс (оценка) проводится на нескольких **бОльших** размерах — вплоть до
10x от обучающего. Это воспроизводит методологию out-of-distribution
генерализации из статьи (Section 5.3, Table 4: там обучение на
`|V| ∈ {10,13,16,19,20}`, тест на 2x-50x от максимального размера обучения),
но со строго одной обучающей размерностью, как и требовалось.

Первая ячейка кода клонирует ветку `gnarl` репозитория и ставит зависимости —
рассчитана на запуск в Google Colab с чистого листа.


In [1]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules

REPO_URL = 'https://github.com/artlvruran/reasoning.git'
REPO_DIR = 'reasoning'
BRANCH = 'gnarl'

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        !git clone --branch $BRANCH --single-branch $REPO_URL $REPO_DIR
    else:
        !cd $REPO_DIR && git fetch origin $BRANCH && git checkout $BRANCH && git pull origin $BRANCH
    !pip install -q jax optax networkx matplotlib pandas pulp
    sys.path.insert(0, REPO_DIR)
else:
    # Локальный запуск: предполагается, что ноутбук уже лежит внутри
    # чекаута репозитория (ветка gnarl).
    sys.path.insert(0, '.')

print('Repo ready, sys.path[0] =', sys.path[0])


Repo ready, sys.path[0] = .


In [2]:
import time
import numpy as np
import jax
import pandas as pd

from gnarl.model import GNARLConfig, init_params, forward
from gnarl.utils import random_euclidean_tsp
from gnarl.envs import TSPEnv, held_karp
from gnarl.training import collect_bc_dataset, train_bc, train_ppo

np.random.seed(0)
print('JAX backend:', jax.default_backend())


JAX backend: cpu


## 1. Обучающие данные и модель: один размер графа

`TSP_TRAIN_N` — единственная размерность (число вершин), на которой
происходит всё обучение (и BC, и PPO). Ниже, на шаге оценки, размер
тестовых графов будет варьироваться от `TSP_TRAIN_N` до `10 * TSP_TRAIN_N`.


In [3]:
def nearest_neighbour_tour(G, start=0):
    n = G.number_of_nodes()
    visited = {start}
    tour = [start]
    cur = start
    for _ in range(n - 1):
        best, best_d = None, float('inf')
        for v in G.nodes():
            if v in visited:
                continue
            d = G[cur][v]['weight'] if G.has_edge(cur, v) else float('inf')
            if d < best_d:
                best_d, best = d, v
        tour.append(best)
        visited.add(best)
        cur = best
    return tour


def two_opt(G, tour, max_passes=40):
    """Local-search improvement of a tour; used as a stand-in for a
    near-optimal reference on sizes where the exact Held-Karp DP is no
    longer tractable (see evaluation section below)."""
    tour = list(tour)
    n = len(tour)

    def dist(u, v):
        return G[u][v]['weight'] if G.has_edge(u, v) else 1e9

    for _ in range(max_passes):
        improved = False
        for i in range(1, n - 1):
            a = tour[i - 1]
            for j in range(i + 1, n):
                b, c, d = tour[i], tour[j], tour[(j + 1) % n]
                delta = (dist(a, c) + dist(b, d)) - (dist(a, b) + dist(c, d))
                if delta < -1e-9:
                    tour[i:j + 1] = tour[i:j + 1][::-1]
                    improved = True
        if not improved:
            break
    return tour

TSP_TRAIN_N = 10  # единственная размерность обучения

t0 = time.time()
tsp_train_graphs = []
for i in range(30):
    G, _ = random_euclidean_tsp(TSP_TRAIN_N, seed=1000 + i)
    tsp_train_graphs.append(G)

def tsp_expert_setup(env, G):
    tour = held_karp(G, start=0)
    env.set_expert_tour(tour)

bc_data_tsp = collect_bc_dataset(lambda: TSPEnv(), tsp_train_graphs, expert_setup=tsp_expert_setup,
                                  reset_kwargs_fn=lambda G: {'start': 0})

cfg_tsp = GNARLConfig(node_specs=TSPEnv.node_specs, edge_specs=TSPEnv.edge_specs, graph_specs=TSPEnv.graph_specs,
                       embed_dim=32, num_mp_layers=3, pooling='max', aggregation='max', use_critic=True)

params_tsp_bc, hist_tsp_bc = train_bc(cfg_tsp, bc_data_tsp, jax.random.PRNGKey(6), epochs=60, lr=2e-3, batch_size=16)
print(f'TSP BC: {len(bc_data_tsp)} examples, all |V|={TSP_TRAIN_N}, {time.time()-t0:.1f}s')


  [BC] epoch 1/60  loss=0.6942  (19 minibatches, 300 examples)
  [BC] epoch 7/60  loss=0.3509  (19 minibatches, 300 examples)
  [BC] epoch 13/60  loss=0.3389  (19 minibatches, 300 examples)
  [BC] epoch 19/60  loss=0.2798  (19 minibatches, 300 examples)
  [BC] epoch 25/60  loss=0.2313  (19 minibatches, 300 examples)
  [BC] epoch 31/60  loss=0.2730  (19 minibatches, 300 examples)
  [BC] epoch 37/60  loss=0.2162  (19 minibatches, 300 examples)
  [BC] epoch 43/60  loss=0.2147  (19 minibatches, 300 examples)
  [BC] epoch 49/60  loss=0.2332  (19 minibatches, 300 examples)
  [BC] epoch 55/60  loss=0.1771  (19 minibatches, 300 examples)
  [BC] epoch 60/60  loss=0.1747  (19 minibatches, 300 examples)
TSP BC: 300 examples, all |V|=10, 24.5s


## 2. PPO дообучение (тот же единственный размер)


In [4]:
t0 = time.time()

def tsp_sampler():
    G, _ = random_euclidean_tsp(TSP_TRAIN_N, seed=int(np.random.randint(0, 1 << 30)))
    return G

params_tsp_ppo, hist_tsp_ppo = train_ppo(
    cfg_tsp, lambda: TSPEnv(), tsp_sampler, jax.random.PRNGKey(7),
    n_updates=20, episodes_per_update=6, ppo_epochs=3, lr=1e-3, ent_coef=0.01,
    reset_kwargs_fn=lambda G: {'start': 0},
)
print(f'TSP PPO: trained only on |V|={TSP_TRAIN_N}, {time.time()-t0:.1f}s')


  [PPO] update 1/20  mean_episode_reward=-5.5524
  [PPO] update 3/20  mean_episode_reward=-5.3737
  [PPO] update 5/20  mean_episode_reward=-5.0389
  [PPO] update 7/20  mean_episode_reward=-4.8459
  [PPO] update 9/20  mean_episode_reward=-3.6046
  [PPO] update 11/20  mean_episode_reward=-2.8499
  [PPO] update 13/20  mean_episode_reward=-3.4194
  [PPO] update 15/20  mean_episode_reward=-3.9302
  [PPO] update 17/20  mean_episode_reward=-4.5716
  [PPO] update 19/20  mean_episode_reward=-4.0578
  [PPO] update 20/20  mean_episode_reward=-3.8304
TSP PPO: trained only on |V|=10, 38.7s


## 3. Инференс на бОльших размерах (OOD generalisation)

Тестовые размеры идут от размера обучения (`1x`) до `10x`. Для `|V| <= 13`
эталон — точный оптимум (Held-Karp DP); для больших `|V|` точный решатель
(в статье — Concorde) недоступен на CPU за разумное время, поэтому эталонным
туром выступает Nearest-Neighbour, улучшенный локальным поиском 2-opt (см.
README основного репозитория, раздел «Осознанные упрощения») — в этих
строках "% above opt" означает "% дороже NN+2-opt тура", близкого к
оптимуму для случайных евклидовых инстансов, но не гарантированно точного.
Ванильный Nearest-Neighbour (без 2-opt) остаётся отдельной, более слабой
эталонной строкой в таблице — именно с ним, а не сам с собой, теперь
сравнивается эталон.


In [5]:
def evaluate_tsp(params, cfg, test_sizes, n_graphs=10, greedy=True):
    pct_above_opt, pct_above_opt_nn = [], []
    for n in test_sizes:
        model_gaps, nn_gaps = [], []
        for i in range(n_graphs):
            G, _ = random_euclidean_tsp(n, seed=50_000 + n * 100 + i)
            opt_tour = held_karp(G, start=0) if n <= 13 else two_opt(G, nearest_neighbour_tour(G, start=0))
            env_ref = TSPEnv()
            env_ref.reset(G, start=0)
            opt_len = env_ref.tour_length(opt_tour)

            env = TSPEnv()
            env.reset(G, start=0)
            done = False
            while not done:
                state = env.state()
                out = forward(params, cfg, state)
                probs = np.asarray(out['probs'])
                a = int(np.argmax(probs)) if greedy else int(np.random.choice(len(probs), p=probs / probs.sum()))
                _, _, done, _ = env.step(a)
            model_len = env.tour_length()
            model_gaps.append(100 * (model_len - opt_len) / opt_len)

            nn_tour = nearest_neighbour_tour(G, start=0)
            nn_len = env_ref.tour_length(nn_tour)
            nn_gaps.append(100 * (nn_len - opt_len) / opt_len)
        pct_above_opt.append(np.mean(model_gaps))
        pct_above_opt_nn.append(np.mean(nn_gaps))
    return pct_above_opt, pct_above_opt_nn

test_sizes_tsp = [TSP_TRAIN_N, 13, 20, 30, 50, 100]
size_labels_tsp = [f'{n} ({n / TSP_TRAIN_N:.1f}x)' for n in test_sizes_tsp]

gaps_bc, gaps_nn = evaluate_tsp(params_tsp_bc, cfg_tsp, test_sizes_tsp)
gaps_ppo, _ = evaluate_tsp(params_tsp_ppo, cfg_tsp, test_sizes_tsp)

table_tsp = pd.DataFrame({
    'Test size |V| (x train size)': size_labels_tsp,
    'Nearest-neighbour (% above opt)': gaps_nn,
    'GNARL_BC  (% above opt)': gaps_bc,
    'GNARL_PPO (% above opt)': gaps_ppo,
})
table_tsp


,Test size |V| (x train size),Nearest-neighbour (% above opt),GNARL_BC (% above opt),GNARL_PPO (% above opt)
0,10 (1.0x),8.632851,3.530679,54.090137
1,13 (1.3x),10.300103,5.045260,60.964602
2,20 (2.0x),13.702790,13.034898,83.722756
3,30 (3.0x),13.206985,30.620072,109.432951
4,50 (5.0x),13.273810,44.464085,151.302658
5,100 (10.0x),16.070315,86.178558,290.202113
